
# 🧠 LangGraph Agent Sandbox

This template demonstrates how to build a **local multi-agent code assistant** using **LangGraph** and **Transformers** — fully compatible with **[Saturn Cloud](https://saturncloud.io/)**.

It uses:
- 🧩 **LangGraph** for workflow control  
- 🤗 **Transformers (Phi-3-mini)** for local code generation  
- 🧮 **LangChain Sandbox** for safe isolated execution (with fallback mode)  

Run this notebook on a Saturn Cloud Jupyter Server to explore autonomous LLM agents that generate, validate, and execute code locally.


## 🧩 Stage 1 — Install Dependencies

In [ ]:
!pip install langchain langchain-community langchain-openai langchain-sandbox

## 🧠 Stage 2 — Load Local Model and Sandbox

In [ ]:

import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_sandbox import PyodideSandbox

model_id = "microsoft/Phi-3-mini-4k-instruct"
print(f"🔧 Loading model: {model_id} ...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)
    llm = HuggingFacePipeline(pipeline=pipe)
    print("✅ Local LLM ready (using Hugging Face Transformers).")
except Exception as e:
    print(f"⚠️ Model load failed: {e}")
    llm = None

try:
    sandbox = PyodideSandbox()
    print("🧪 Pyodide Sandbox ready for isolated execution.")
except Exception as e:
    print(f"⚠️ Sandbox initialization failed: {e}")
    print("➡️ Using lightweight local sandbox emulator (safe fallback).")

    class LocalSandbox:
        def run(self, code):
            try:
                exec_locals = {}
                exec(code, {}, exec_locals)
                return {"output_text": str(exec_locals)}
            except Exception as err:
                return {"output_text": f"Error: {err}"}

    sandbox = LocalSandbox()


## ⚙️ Stage 3 — Define Workflow Agents

In [ ]:

from langgraph.graph import StateGraph
from typing import TypedDict, Dict
import re, io, sys, contextlib

class CodeState(TypedDict):
    prompt: str
    code: str
    check: str
    output: str

class CodeGenerator:
    def run(self, state: CodeState) -> Dict:
        query = f"Write clean, well-commented Python 3 code for: {state['prompt']}"
        response = llm.invoke(query)
        generated = response if isinstance(response, str) else getattr(response, "content", str(response))
        print("🧠 Generated code:")
        print(generated[:300], "..." if len(generated) > 300 else "")
        return {"code": generated}

class SyntaxChecker:
    def run(self, state: CodeState) -> Dict:
        code = state.get("code", "")
        issues = []
        if not re.search(r"def\s+\w+\(", code):
            issues.append("❌ No function definition detected.")
        if code.count("(") != code.count(")"):
            issues.append("⚠️ Unbalanced parentheses.")
        msg = "✅ Syntax check passed." if not issues else " | ".join(issues)
        print(f"🧮 Syntax Check : {msg}")
        return {"check": msg}

class SandboxExecutor:
    def __init__(self, sandbox_instance):
        self.sandbox = sandbox_instance

    def run(self, state: CodeState) -> Dict:
        code = state.get("code", "")
        if not code:
            return {"output": "No code detected."}
        code = re.sub(r"```python|```", "", code).strip()
        buffer = io.StringIO()
        try:
            with contextlib.redirect_stdout(buffer):
                exec_locals = {}
                exec(code, {}, exec_locals)
            return {"output": buffer.getvalue().strip() or "✅ Executed successfully."}
        except Exception as e:
            return {"output": f"Error: {e}"}

def build_workflow():
    g = StateGraph(CodeState)
    g.add_node("generate_code", CodeGenerator().run)
    g.add_node("check_syntax", SyntaxChecker().run)
    g.add_node("execute_code", SandboxExecutor(sandbox).run)
    g.set_entry_point("generate_code")
    g.add_edge("generate_code", "check_syntax")
    g.add_edge("check_syntax", "execute_code")
    print("✅ LangGraph workflow built.")
    return g.compile()

workflow = build_workflow()


## 🧪 Stage 4 — Batch Testing

In [ ]:

examples = [
    "Write a Python function to check if a number is prime.",
    "Create a Python script that sorts a list of strings alphabetically.",
    "Generate a function to calculate factorial using recursion."
]
for i, prompt in enumerate(examples, start=1):
    print(f"\n🧠 Test {i}: {prompt}")
    print("=" * 80)
    result = workflow.invoke({"prompt": prompt, "code": "", "check": "", "output": ""})
    print(result)
    print("=" * 80)


## 💬 Stage 5 — Interactive Assistant

In [ ]:

from rich.console import Console
from rich.panel import Panel

console = Console()
def run_interactive():
    console.print(Panel.fit("💬 Type a coding task (or 'exit' to quit)", style="bold cyan"))
    while True:
        prompt = input("\n🧠 Enter task: ").strip()
        if prompt.lower() in ["exit", "quit", "q"]:
            console.print("\n👋 Exiting interactive mode.\n", style="bold yellow")
            break
        state = {"prompt": prompt, "code": "", "check": "", "output": ""}
        result = workflow.invoke(state)
        console.print(result)
run_interactive()



## 🏁 Conclusion

You’ve built a **LangGraph Agent Sandbox** — a self-contained multi-agent system that generates, checks, and executes Python code using local models.  
All this runs **directly inside Saturn Cloud**, without external APIs.

**Built with ❤️ using:**  
🤗 Transformers | 🧮 LangGraph | ⚡ LangChain | ☁️ [Saturn Cloud](https://saturncloud.io/)
